# NLP Mini Project 1 — Wikipedia Article-Type Classifier (DBpedia-14)

**Course:** Natural Language Processing — Phase 1 (Units 1–4)
**Corpus option:** Option 6 — Wikipedia-based articles (DBpedia-14)
**Task:** Given the text of a Wikipedia article abstract, predict what kind of entity it describes.

### Problem statement (Unit 1)
Wikipedia contains millions of articles, and many applications (search filters, knowledge-base construction,
content recommendation, automatic tagging of new pages) need to know *what type of thing* an article is about.
Doing this by hand does not scale. This project builds a classical NLP pipeline that reads an article abstract
and assigns one of six types: **Company, Artist, Athlete, Building, Film, WrittenWork**.

### Program flow
Corpus → Train/Test separation → Preprocessing → Corpus statistics → N-gram analysis →
Next-word prediction → Bag of Words → Classical classifier → Test metrics → Demo prediction

## 0. Setup

In [ ]:
# In Google Colab, first run:  !pip install datasets nltk scikit-learn pandas matplotlib seaborn joblib
import os, sys, json, random, time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

PROJECT_DIR = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
from preprocessing import preprocess, clean_text, sentences_for_lm, STOP_WORDS
from ngram_lm import NGramLM

DATA_DIR    = os.path.join(PROJECT_DIR, 'data')
MODEL_DIR   = os.path.join(PROJECT_DIR, 'models')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
FIG_DIR     = os.path.join(RESULTS_DIR, 'figures')
for d in (DATA_DIR, MODEL_DIR, RESULTS_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid')
print('Project dir:', PROJECT_DIR)

## 1. Corpus acquisition and understanding

**DBpedia-14** (Zhang, Zhao & LeCun, 2015) contains Wikipedia abstracts extracted through DBpedia, labelled with
14 non-overlapping ontology classes. Each record has `title`, `content` (the abstract) and `label`.
The official split has 560,000 training and 70,000 test examples (40,000 / 5,000 per class).

**Design choices (documented as required):**
- **Classes:** 6 of the 14 — Company, Artist, Athlete, Building, Film, WrittenWork. They cover organisations,
  people, places and creative works, and include pairs that are expected to be confusable (Film vs WrittenWork,
  Artist vs Film/WrittenWork, Company vs Building).
- **Text field:** *content only*. The title is just a name (e.g. "E. D. Abbott Ltd") and usually reappears at the
  start of the content, so it adds little; using the content alone makes the task depend on language, not names.
- **Training data:** a balanced, reproducible subset of **3,000 articles per class (18,000)** sampled from the
  official training split with `random_state=42`.
- **Test data:** **all** official test examples of the 6 classes (**30,000**, 5,000 per class). They are never used
  for building the vocabulary, model selection or tuning — only for the final evaluation.

In [ ]:
SELECTED_CLASSES = ['Company', 'Artist', 'Athlete', 'Building', 'Film', 'WrittenWork']
TRAIN_PER_CLASS = 3000
TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
TEST_CSV  = os.path.join(DATA_DIR, 'test.csv')

if os.path.exists(TRAIN_CSV) and os.path.exists(TEST_CSV):
    # Frozen split already saved: always reuse it so results are reproducible.
    train_df = pd.read_csv(TRAIN_CSV)
    test_df  = pd.read_csv(TEST_CSV)
    print('Loaded frozen split from data/')
else:
    from datasets import load_dataset
    ds = load_dataset('fancyzhx/dbpedia_14')
    names = ds['train'].features['label'].names

    def to_frame(split):
        df = ds[split].to_pandas().reset_index().rename(columns={'index': 'orig_idx'})
        df['label_name'] = df['label'].map(lambda i: names[i])
        df['text'] = df['content'].str.strip()
        return df[df['label_name'].isin(SELECTED_CLASSES)][['orig_idx', 'title', 'text', 'label_name']]

    train_full, test_full = to_frame('train'), to_frame('test')
    train_df = (train_full.groupby('label_name').sample(n=TRAIN_PER_CLASS, random_state=SEED)
                .sample(frac=1, random_state=SEED).reset_index(drop=True))
    test_df = test_full.reset_index(drop=True)          # official test split, untouched
    train_df.to_csv(TRAIN_CSV, index=False)
    test_df.to_csv(TEST_CSV, index=False)
    print('Created and saved the frozen split to data/train.csv and data/test.csv')

print('Train:', train_df.shape, ' Test:', test_df.shape)
train_df.head()

In [ ]:
# Sanity check: no article text appears in both train and test
overlap = set(train_df.text) & set(test_df.text)
print('Texts shared between train and test:', len(overlap))
print('\nOne example per class (train):')
for c in SELECTED_CLASSES:
    row = train_df[train_df.label_name == c].iloc[0]
    print(f'\n[{c}] {row.title}\n  {row.text[:220]}...')

## 2. Text preprocessing (Unit 2)

One reusable function, `preprocess()` in `src/preprocessing.py`, is used for statistics, N-grams, the classifier
and the demo:

1. **Normalization** — lowercase, remove URLs, digits and punctuation, collapse whitespace.
2. **Tokenization** — NLTK `word_tokenize`.
3. **Stop-word removal** — NLTK English stop-word list (179 words).
4. **Lemmatization** — WordNet lemmatizer (chosen over stemming because it keeps real words, which makes the
   N-gram tables and classifier features readable; a stemming comparison is shown below).
5. Tokens shorter than 2 characters are dropped.

For the **language model** a lighter version (`sentences_for_lm`) keeps stop words and word forms, because
function words ("was", "the", "by") are exactly what a next-word predictor must learn.

In [ ]:
examples = train_df.groupby('label_name').head(1).text.tolist()[:3]
for t in examples:
    print('ORIGINAL :', t[:200])
    print('CLEANED  :', clean_text(t)[:200])
    print('LEMMA    :', preprocess(t)[:25])
    print('STEM     :', preprocess(t, normalize='stem')[:25])
    print('-' * 100)

In [ ]:
t0 = time.time()
train_df['tokens'] = train_df.text.apply(preprocess)
test_df['tokens']  = test_df.text.apply(preprocess)
train_df['clean']  = train_df.tokens.str.join(' ')
test_df['clean']   = test_df.tokens.str.join(' ')
print(f'Preprocessed {len(train_df) + len(test_df):,} documents in {time.time() - t0:.1f}s')
train_df[['label_name', 'text', 'clean']].head(5)

## 3. Corpus statistics (training split)

In [ ]:
raw_tokens = train_df.text.apply(lambda t: clean_text(t).split())
stats = {
    'documents (train)': len(train_df),
    'documents (test)': len(test_df),
    'classes': len(SELECTED_CLASSES),
    'total raw tokens (train)': int(raw_tokens.str.len().sum()),
    'vocabulary size, raw lowercase (train)': len(set(w for ts in raw_tokens for w in ts)),
    'total tokens after preprocessing (train)': int(train_df.tokens.str.len().sum()),
    'vocabulary size after preprocessing (train)': len(set(w for ts in train_df.tokens for w in ts)),
    'avg raw tokens per document': round(raw_tokens.str.len().mean(), 1),
    'avg tokens per document after preprocessing': round(train_df.tokens.str.len().mean(), 1),
}
pd.Series(stats, name='value').to_csv(os.path.join(RESULTS_DIR, 'corpus_statistics.csv'), header=['value'], index_label='statistic')
pd.Series(stats, name='value').to_frame()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pd.DataFrame({'train': train_df.label_name.value_counts(), 'test': test_df.label_name.value_counts()}) \
  .loc[SELECTED_CLASSES].plot.bar(ax=axes[0], rot=20, title='Class distribution')
train_df.assign(n=raw_tokens.str.len()).boxplot(column='n', by='label_name', ax=axes[1], rot=20)
axes[1].set_title('Document length (raw tokens) by class'); axes[1].set_xlabel(''); plt.suptitle('')
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'class_distribution_length.png'), dpi=120); plt.show()
train_df.assign(n=raw_tokens.str.len()).groupby('label_name').n.describe().round(1)

In [ ]:
word_freq = Counter(w for ts in train_df.tokens for w in ts)
top = pd.DataFrame(word_freq.most_common(25), columns=['word', 'count'])
top.plot.barh(x='word', y='count', figsize=(8, 7), legend=False, title='Top 25 words (after preprocessing)').invert_yaxis()
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'top_words.png'), dpi=120); plt.show()

hapax = sum(1 for c in word_freq.values() if c == 1)
print(f'Words that occur only once (hapax legomena): {hapax:,} ({hapax / len(word_freq):.1%} of the vocabulary)')

**Observations**
- The classes are perfectly balanced (3,000 train / 5,000 test each), so accuracy is a meaningful metric and macro
  averages equal weighted averages.
- Documents are short abstracts (around 45–50 raw tokens on average), with few very long outliers.
- Preprocessing roughly halves the number of tokens (stop words removed) and reduces the vocabulary through
  lowercasing and lemmatization.
- The most frequent words are already strongly topical (*film, born, company, directed, published, historic, book, novel*),
  which suggests a Bag-of-Words model will work well. Roughly half of the vocabulary appears only once
  (Zipf's law) — mostly proper nouns — so rare words are filtered with `min_df=2` in the classifier.

## 4. N-gram analysis (Unit 3)

In [ ]:
from nltk.util import ngrams

def top_ngrams(token_lists, n, k=15):
    c = Counter()
    for ts in token_lists:
        c.update(ngrams(ts, n))
    return [(' '.join(g), cnt) for g, cnt in c.most_common(k)]

overall = {f'{n}-gram': top_ngrams(train_df.tokens, n) for n in (1, 2, 3)}
pd.DataFrame([{'n': k, 'ngram': g, 'count': c} for k, v in overall.items() for g, c in v]) \
  .to_csv(os.path.join(RESULTS_DIR, 'top_ngrams.csv'), index=False)
pd.DataFrame({k: [f'{g} ({c})' for g, c in v] for k, v in overall.items()})

In [ ]:
def per_class_table(n, k=10):
    return pd.DataFrame({c: [f'{g} ({cnt})' for g, cnt in top_ngrams(train_df[train_df.label_name == c].tokens, n, k)]
                         for c in SELECTED_CLASSES})
for n, name in [(1, 'unigrams'), (2, 'bigrams'), (3, 'trigrams')]:
    pd.DataFrame([{'class': c, 'rank': r + 1, 'ngram': g, 'count': cnt}
                  for c in SELECTED_CLASSES
                  for r, (g, cnt) in enumerate(top_ngrams(train_df[train_df.label_name == c].tokens, n, 15))]) \
      .to_csv(os.path.join(RESULTS_DIR, f'class_{name}.csv'), index=False)
print('Top bigrams per class'); display(per_class_table(2))
print('Top trigrams per class'); display(per_class_table(3))

In [ ]:
# Distinctive bigrams: frequent in one class but rare in the others (ratio with add-1 smoothing)
bigram_counts = {c: Counter(g for ts in train_df[train_df.label_name == c].tokens for g in ngrams(ts, 2))
                 for c in SELECTED_CLASSES}
distinct, distinct_rows = {}, []
for c in SELECTED_CLASSES:
    others = Counter()
    for o in SELECTED_CLASSES:
        if o != c: others.update(bigram_counts[o])
    scored = [(g, cnt, cnt / (others[g] + 1)) for g, cnt in bigram_counts[c].items() if cnt >= 20]
    scored.sort(key=lambda x: -x[2])
    distinct[c] = [f"{' '.join(g)} ({cnt})" for g, cnt, _ in scored[:10]]
    for g, cnt, ratio in scored[:15]:
        distinct_rows.append({'class': c, 'bigram': ' '.join(g), 'count_in_class': cnt,
                              'count_in_other_classes': others[g], 'ratio': round(ratio, 2)})
pd.DataFrame(distinct_rows).to_csv(os.path.join(RESULTS_DIR, 'characteristic_bigrams.csv'), index=False)
pd.DataFrame(distinct)

In [ ]:
# Bigrams shared by several classes — sources of possible confusion
shared = []
for g in set().union(*[set(dict(bigram_counts[c].most_common(300))) for c in SELECTED_CLASSES]):
    present = [c for c in SELECTED_CLASSES if bigram_counts[c][g] >= 15]
    if len(present) >= 2:
        shared.append((' '.join(g), ', '.join(f'{c}:{bigram_counts[c][g]}' for c in present)))
pd.DataFrame(sorted(shared, key=lambda x: -sum(int(p.split(':')[1]) for p in x[1].split(', ')))[:15],
             columns=['bigram', 'counts per class'])

**What the N-grams reveal**
- **Unigrams** show the topic vocabulary of each type (film, company, album, player, building, novel).
- **Bigrams/trigrams** capture fixed Wikipedia phrasing that is highly class-specific: *football player*,
  *professional ice hockey*, *directed by*, *national register historic place*, *record label*, *new york times
  best seller*, *founded in*, *headquartered in*. These multi-word cues are more precise than single words.
- **Distinctive bigrams** (frequent in one class, rare elsewhere) are almost perfect indicators of their class,
  explaining why a simple bag-of-words classifier is very accurate on this corpus.
- **Shared bigrams** such as *united state(s)*, *new york*, *also known* or *th century* (from "19th century"
  after digits are removed) occur in almost every class; they describe places and dates, not the entity type, and
  carry little class information. *born + month* is shared by the two "person" classes (Artist and Athlete), so
  the classifier must rely on other words (*painter*, *singer* vs *footballer*, *league*) to separate them.
- Lemmatization merges *united states* into *united state*, a small side effect of word-level normalization.

## 5. N-gram language model and next-word prediction (Unit 3)

A **bigram** and a **trigram** model are trained on sentences of the *training* articles only (stop words kept).
Sentences get `<s>` / `</s>` markers. Unseen N-grams must not get zero probability, so two smoothing methods
are compared using **perplexity on unseen test sentences** (2,000 test articles):

- **Add-k smoothing** (k = 0.01): add a small count to every possible N-gram.
- **Linear interpolation**: mix the trigram, bigram and unigram estimates,
  $P(w \mid a,b) = \lambda_3 P_{ML}(w \mid a,b) + \lambda_2 P_{ML}(w \mid b) + \lambda_1 P_{add1}(w)$
  (bigram model: $\lambda_2 P_{ML}(w \mid b) + \lambda_1 P_{add1}(w)$). The weights $\lambda$ are chosen on a
  **validation set taken from the training articles (10%)** — never on the test set.

For next-word prediction the model uses the trigram context when it has been seen and **backs off** to the bigram
(then unigram) otherwise. Words never seen in training are skipped when computing perplexity.

In [ ]:
lm_train_sents = [s for t in train_df.text for s in sentences_for_lm(t)]
lm_test_sents  = [s for t in test_df.sample(2000, random_state=SEED).text for s in sentences_for_lm(t)]
print(f'LM training sentences: {len(lm_train_sents):,}   held-out test sentences: {len(lm_test_sents):,}')

# 1) Choose interpolation weights on a validation slice of the TRAINING articles
val_idx = train_df.sample(frac=0.1, random_state=SEED).index
fit_sents = [s for t in train_df.drop(val_idx).text for s in sentences_for_lm(t)]
val_sents = [s for t in train_df.loc[val_idx].text for s in sentences_for_lm(t)]
grids = {2: [(0.8, 0.2), (0.7, 0.3), (0.6, 0.4), (0.5, 0.5)],
         3: [(0.6, 0.3, 0.1), (0.4, 0.4, 0.2), (0.3, 0.5, 0.2), (0.2, 0.5, 0.3), (0.1, 0.6, 0.3)]}
best_lambdas, tuning_rows = {}, []
for n, grid in grids.items():
    lm = NGramLM(n=n).fit(fit_sents)
    for lam in grid:
        lm.lambdas = lam
        tuning_rows.append({'model': f'{n}-gram', 'lambdas': lam, 'validation perplexity': round(lm.perplexity(val_sents), 1)})
    best = min((r for r in tuning_rows if r['model'] == f'{n}-gram'), key=lambda r: r['validation perplexity'])
    best_lambdas[n] = best['lambdas']
pd.DataFrame(tuning_rows).to_csv(os.path.join(RESULTS_DIR, 'lm_lambda_tuning.csv'), index=False)
display(pd.DataFrame(tuning_rows))
print('Selected lambdas:', best_lambdas)

# 2) Fit the final models on all training sentences and evaluate once on unseen test sentences
lms = {(n, sm): NGramLM(n=n, smoothing=sm, lambdas=best_lambdas[n]).fit(lm_train_sents)
       for n in (2, 3) for sm in ('addk', 'interp')}
bigram_lm, trigram_lm = lms[(2, 'interp')], lms[(3, 'interp')]
print(f'Vocabulary size: {trigram_lm.V:,}')
print(f'P_ML(by | directed)       = {trigram_lm.bigrams[("directed", "by")] / trigram_lm.unigrams["directed"]:.3f}')
print(f'P_ML(player | football)   = {trigram_lm.bigrams[("football", "player")] / trigram_lm.unigrams["football"]:.3f}')
print(f'P_ML(in | was founded)    = {trigram_lm.trigrams[("was", "founded", "in")] / trigram_lm.bigrams[("was", "founded")]:.3f}')

ppl = pd.DataFrame([{'model': f'{"bigram" if n == 2 else "trigram"}', 'smoothing': 'add-k (k=0.01)' if sm == 'addk' else 'interpolation',
                     'test perplexity': round(lm.perplexity(lm_test_sents), 1)} for (n, sm), lm in lms.items()])
ppl.to_csv(os.path.join(RESULTS_DIR, 'lm_perplexity.csv'), index=False)
ppl

In [ ]:
prompts = ['the film was directed', 'he is a professional', 'the company was founded',
           'the novel was written', 'the building is located', 'she is an american',
           'the airline is based', 'the church was built in']
rows = []
for p in prompts:
    toks = clean_text(p).split()
    preds, used = trigram_lm.predict_next(toks, top_k=5)
    rows.append({'prompt': p, 'context used': used,
                 'top-5 next words (probability)': ', '.join(f'{w} ({pr:.2f})' for w, pr in preds),
                 'greedy continuation': trigram_lm.generate(toks, max_words=8)})
pd.DataFrame(rows).to_csv(os.path.join(RESULTS_DIR, 'next_word_predictions.csv'), index=False)
pd.DataFrame(rows)

**Interpretation**
- The model learned typical Wikipedia phrasing: *"the film was directed" → by*, *"he is a professional" →
  football/baseball/ice…*, *"the company was founded" → in*. The predictions also reveal the entity type, which
  links the language model to the classification task.
- **Smoothing matters as much as the order of the model.** With add-k smoothing the trigram is *worse* than the
  bigram: most trigram contexts are rare, so adding k to all ~60,000 possible next words gives far too much
  probability to unseen words. With interpolation the trigram is better than the bigram, because it uses the
  extra context when it exists and falls back on bigram/unigram evidence when it does not.
- The tuned weights give the trigram estimate a *small* share (λ₃ ≈ 0.2): with under a million training tokens
  most trigrams are seen only once or never, so the model learns to trust the bigram more.
- This shows the sparsity trade-off from Unit 3: a larger *n* gives more context but many more unseen N-grams.
- Greedy generation quickly becomes generic or repetitive: N-gram models only see 1–2 previous words.
  Because numbers were removed in preprocessing, phrases such as *"founded in 1990"* become *"founded in"*,
  which also shows up in the continuations. These limitations are addressed by neural and Transformer language
  models in Project 2.

## 6. Bag of Words + classical classifier (Unit 4)

- **Representation:** `CountVectorizer` (Bag of Words), fitted on the **training texts only**. `min_df=2` removes
  words seen in a single document (mostly names), keeping the vocabulary manageable.
- **Classifiers:** Multinomial Naive Bayes and Logistic Regression.
- **Model selection on a validation set:** 10% of the training articles (1,800, stratified, `random_state=42`) are
  held out. Each configuration — classifier × features (single words, or words + word pairs) × settings — is trained
  on the other 90% and scored on the validation set. The best configuration is then retrained on all 18,000
  training articles. The test set is not used for any of these choices.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

X_train, y_train = train_df.clean, train_df.label_name
X_test,  y_test  = test_df.clean,  test_df.label_name
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.10, stratify=y_train, random_state=SEED)
print(f'Fit: {len(X_fit):,}  Validation: {len(X_val):,}  (both from the training set)')

def make_pipeline(model_name, ngram_range, param):
    clf = MultinomialNB(alpha=param) if model_name == 'Naive Bayes' else LogisticRegression(max_iter=3000, C=param)
    return Pipeline([('bow', CountVectorizer(min_df=2, ngram_range=ngram_range)), ('clf', clf)])

grid = ([('Naive Bayes', ng, a) for ng in [(1, 1), (1, 2)] for a in [0.1, 0.5, 1.0]] +
        [('Logistic Regression', ng, C) for ng in [(1, 1), (1, 2)] for C in [0.1, 0.5, 1.0]])
val_rows = []
for model_name, ng, param in grid:
    pipe = make_pipeline(model_name, ng, param).fit(X_fit, y_fit)
    pred = pipe.predict(X_val)
    val_rows.append({'model': model_name, 'features': 'words' if ng == (1, 1) else 'words + word pairs',
                     'setting': f"{'alpha' if model_name == 'Naive Bayes' else 'C'}={param}",
                     'n_features': len(pipe.named_steps['bow'].vocabulary_),
                     'val_accuracy': accuracy_score(y_val, pred), 'val_macro_f1': f1_score(y_val, pred, average='macro'),
                     '_cfg': (model_name, ng, param)})
val_table = pd.DataFrame(val_rows).sort_values('val_accuracy', ascending=False).reset_index(drop=True)
val_table.drop(columns='_cfg').round(4).to_csv(os.path.join(RESULTS_DIR, 'validation_experiments.csv'), index=False)
best_cfg = val_table.iloc[0]['_cfg']
best_name = best_cfg[0]
print('Selected on validation:', best_cfg)
val_table.drop(columns='_cfg').round(4)

In [ ]:
model = make_pipeline(*best_cfg).fit(X_train, y_train)      # retrain on all 18,000 training articles
vectorizer = model.named_steps['bow']
X_bow = vectorizer.transform(X_train)
print(f'BoW matrix (train): {X_bow.shape[0]:,} documents x {X_bow.shape[1]:,} features, '
      f'{X_bow.nnz / np.prod(X_bow.shape):.3%} non-zero (very sparse)')
print('Example vector (first document), non-zero entries:')
row = X_bow[0]
print({vectorizer.get_feature_names_out()[j]: int(v) for j, v in zip(row.indices[:15], row.data[:15])})

joblib.dump(model, os.path.join(MODEL_DIR, 'bow_classifier.joblib'))            # full pipeline (used by demo.py)
joblib.dump(vectorizer, os.path.join(MODEL_DIR, 'bow_vectorizer.joblib'))       # CountVectorizer alone
joblib.dump(model.named_steps['clf'], os.path.join(MODEL_DIR, 'classifier.joblib'))  # classifier alone
print('Saved models/bow_classifier.joblib, models/bow_vectorizer.joblib, models/classifier.joblib')

## 7. Final evaluation on unseen test data

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
print(f'Model: {best_name}   |   test documents: {len(y_test):,}')
print(f'Accuracy {acc:.4f} | Macro precision {p:.4f} | Macro recall {r:.4f} | Macro F1 {f1:.4f}\n')
report = classification_report(y_test, y_pred, labels=SELECTED_CLASSES, digits=4)
print(report)

metrics = {'model': best_name, 'test_size': int(len(y_test)), 'accuracy': acc, 'macro_precision': p,
           'macro_recall': r, 'macro_f1': f1,
           'per_class': classification_report(y_test, y_pred, labels=SELECTED_CLASSES, output_dict=True)}
with open(os.path.join(RESULTS_DIR, 'test_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
pd.DataFrame(metrics['per_class']).T.loc[SELECTED_CLASSES].round(4) \
  .to_csv(os.path.join(RESULTS_DIR, 'per_class_metrics.csv'), index_label='class')
config = {'model': best_cfg[0], 'features': 'CountVectorizer (Bag of Words)', 'ngram_range': list(best_cfg[1]),
          'min_df': 2, ('alpha' if best_cfg[0] == 'Naive Bayes' else 'C'): best_cfg[2],
          'preprocessing': 'lowercase, strip accents, remove URLs/digits/punctuation, NLTK tokenize, stop words removed, WordNet lemma',
          'classes': SELECTED_CLASSES, 'train_size': int(len(train_df)), 'test_size': int(len(test_df)),
          'seed': SEED, 'test_accuracy': round(acc, 4), 'test_macro_f1': round(f1, 4)}
with open(os.path.join(MODEL_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

# For reference only (after the choice was made): the best Naive Bayes configuration on the same test set
best_nb = val_table[val_table.model == 'Naive Bayes'].iloc[0]['_cfg']
nb_pred = make_pipeline(*best_nb).fit(X_train, y_train).predict(X_test)
comparison = pd.DataFrame([
    {'model': f'{best_cfg[0]} (selected)', 'features': val_table.iloc[0]['features'], 'test_accuracy': acc, 'test_macro_f1': f1},
    {'model': 'Naive Bayes', 'features': val_table[val_table.model == 'Naive Bayes'].iloc[0]['features'],
     'test_accuracy': accuracy_score(y_test, nb_pred), 'test_macro_f1': f1_score(y_test, nb_pred, average='macro')}]).round(4)
comparison.to_csv(os.path.join(RESULTS_DIR, 'test_comparison.csv'), index=False)
display(comparison)

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=SELECTED_CLASSES)
plt.figure(figsize=(7.5, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=SELECTED_CLASSES, yticklabels=SELECTED_CLASSES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'Confusion matrix — {best_name} (test set)')
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'confusion_matrix.png'), dpi=120); plt.show()

pairs = [(SELECTED_CLASSES[i], SELECTED_CLASSES[j], cm[i, j]) for i in range(6) for j in range(6) if i != j]
pd.DataFrame(sorted(pairs, key=lambda x: -x[2])[:8], columns=['true', 'predicted', 'count'])

In [ ]:
proba = model.predict_proba(X_test)
test_df['predicted'] = y_pred
test_df['confidence'] = proba.max(axis=1)
test_df['correct'] = test_df.predicted == test_df.label_name
test_df[['orig_idx', 'title', 'label_name', 'predicted', 'confidence', 'correct']] \
    .to_csv(os.path.join(RESULTS_DIR, 'test_predictions.csv'), index=False)

print('15 individual test predictions (random sample, seed 42):')
sample = test_df.sample(15, random_state=SEED)
sample.assign(text=sample.text.str[:110])[['title', 'text', 'label_name', 'predicted', 'confidence', 'correct']].round(3)

### Error analysis

In [ ]:
errors = test_df[~test_df.correct]
errors[['orig_idx', 'title', 'label_name', 'predicted', 'confidence', 'text']].round(3) \
  .to_csv(os.path.join(RESULTS_DIR, 'misclassified_examples.csv'), index=False)
print(f'Misclassified: {len(errors):,} of {len(test_df):,} ({len(errors) / len(test_df):.2%})')
print(f'Mean confidence — correct: {test_df[test_df.correct].confidence.mean():.3f}, wrong: {errors.confidence.mean():.3f}\n')
for _, row in errors.sample(8, random_state=SEED).iterrows():
    print(f'[{row.label_name} -> predicted {row.predicted}, conf {row.confidence:.2f}] {row.title}')
    print('   ', row.text[:260], '\n')

In [ ]:
# Most influential words per class (what the model actually learned)
clf = model.named_steps['clf']
feats = vectorizer.get_feature_names_out()
if hasattr(clf, 'coef_'):
    weights = clf.coef_
else:  # Naive Bayes: log P(word|class) minus the average over classes
    weights = clf.feature_log_prob_ - clf.feature_log_prob_.mean(axis=0)
pd.DataFrame([{'class': c, 'rank': r + 1, 'word_or_pair': feats[j], 'weight': round(float(weights[i][j]), 3)}
              for i, c in enumerate(clf.classes_) for r, j in enumerate(np.argsort(weights[i])[::-1][:20])]) \
  .to_csv(os.path.join(RESULTS_DIR, 'top_features.csv'), index=False)
pd.DataFrame({c: [feats[j] for j in np.argsort(weights[i])[::-1][:12]] for i, c in enumerate(clf.classes_)})

**Interpretation of the results**
- The classifier is highly accurate on 30,000 unseen articles; precision and recall are high and balanced for all
  six classes, so no class is being sacrificed for another.
- The most influential words per class match the distinctive N-grams found in Section 4 (e.g. *film, directed*
  for Film; *novel, book, published* for WrittenWork; *player, league, played* for Athlete).
- **Where errors happen** (see the confusion matrix and examples): mostly between semantically close types —
  **Artist → WrittenWork** (writers, cartoonists and memoir authors are described with *book, published, novel*),
  **Company ↔ Building** (restaurant chains, hotels, shops and factories are both businesses and places), and
  Artist → Company (designers and producers described through the companies they worked for).
  Athlete and Film are the easiest classes: their vocabulary (*footballer, league, played* / *directed, starring*)
  is used by almost no other class.
- Many errors are **words pointing to the wrong type**: the Catholic periodical *Catholic World* mentions a
  *church* and a *father* and is predicted as Building; the comic book *American Vampire* is described through the
  *writer* who created it and is predicted as Artist; the *Arden* estate (Building) is described through the
  *railroad magnate* who owned it and is predicted as Company. Bag of Words counts the words but cannot tell
  *what the article is about* from *what it mentions*.
- Some errors are **label noise**: *Wea*, a Native American tribe, is labelled Company in DBpedia — no model
  could get this "right".
- Wrong predictions have clearly lower confidence than correct ones, so the probability output is useful for
  flagging uncertain cases.
- **Limitations:** Bag of Words ignores word order and meaning ("not a film" vs "a film"); synonyms are treated
  as unrelated features; very short or list-like abstracts give little evidence. These motivate embeddings and
  Transformer models in Project 2.

## 8. Demo — classify new text

In [ ]:
def predict_article_type(text, show=True):
    clean = preprocess(text, as_string=True)
    probs = model.predict_proba([clean])[0]
    ranked = sorted(zip(model.classes_, probs), key=lambda x: -x[1])
    if show:
        print(f'Text: {text[:120]}{"..." if len(text) > 120 else ""}')
        print(f'  -> {ranked[0][0]}  ({ranked[0][1]:.1%})   runner-up: {ranked[1][0]} ({ranked[1][1]:.1%})\n')
    return ranked[0][0]

new_texts = [  # written for this demo, not taken from the corpus
    'Nova Robotics is a technology firm headquartered in Austin Texas that develops warehouse automation software. It was founded in 2015.',
    'Maria Lopez is a Spanish painter and sculptor whose works have been exhibited in Madrid and Paris.',
    'Daniel Okafor is a Nigerian professional footballer who plays as a striker for a club in the English Premier League.',
    'The Harbour Tower is a 40-storey skyscraper completed in 1998 in the city centre and designed by a local architect.',
    'Silent Orbit is a 2019 science fiction thriller directed by James Reed and starring two astronauts stranded in space.',
    'The Glass Garden is a novel by a British author published in 2004 that follows three generations of one family.',
]
for t in new_texts:
    predict_article_type(t)

In [ ]:
# Interactive demo: uncomment and run, then type any article-like text.
# predict_article_type(input('Enter a short article text: '))

## 9. Summary

In [ ]:
summary = pd.Series({
    'Corpus': 'DBpedia-14 (6 classes, content only)',
    'Train / Test documents': f'{len(train_df):,} / {len(test_df):,} (official test split)',
    'Vocabulary after preprocessing (train)': f"{stats['vocabulary size after preprocessing (train)']:,}",
    'BoW features (min_df=2)': f'{len(feats):,}',
    'LM test perplexity, interpolated bigram / trigram': f"{ppl.iloc[1, 2]} / {ppl.iloc[3, 2]}",
    'Selected classifier (validation set)': f'{best_cfg[0]}, {val_table.iloc[0]["features"]}, {val_table.iloc[0]["setting"]}',
    'Test accuracy': f'{acc:.4f}',
    'Test macro F1': f'{f1:.4f}',
}, name='value')
summary.to_frame()

### Files kept for Project 2
| Path | Content |
|---|---|
| `data/train.csv`, `data/test.csv` | Frozen split (with `orig_idx` into the official DBpedia-14 splits) |
| `src/preprocessing.py` | Reusable preprocessing function |
| `src/ngram_lm.py` | N-gram language model |
| `models/bow_classifier.joblib` | Trained pipeline (CountVectorizer + Logistic Regression), used by `demo.py` |
| `models/bow_vectorizer.joblib`, `models/classifier.joblib`, `models/config.json` | The two parts saved separately + the chosen settings |
| `results/*.csv` | Every table in this notebook: statistics, N-grams, LM, validation experiments, metrics, errors |
| `results/figures/` | Figures used in the report and presentation |

**References:** Zhang, X., Zhao, J., LeCun, Y. (2015). *Character-level Convolutional Networks for Text
Classification*. NeurIPS — DBpedia-14 dataset (Hugging Face: `fancyzhx/dbpedia_14`).
Jurafsky & Martin, *Speech and Language Processing* (3rd ed. draft), chapters on N-grams and text classification.
Libraries: NLTK, scikit-learn, pandas, matplotlib, seaborn.